# ECON 662 — Problem Set 2
## Part (d): Linear Regression Approach (Scott 2014)

Estimate $\theta$ without solving the value function in the estimation algorithm and without estimating the replacement-cost process explicitly, using the linear-regression approach discussed in Lecture 15.

This notebook follows the logic of Scott (2014) as adapted to this bus-replacement problem:

1. Use observed data to estimate conditional choice probabilities (CCPs).
2. Use a termination-action argument to turn future value differences into observable CCP differences.
3. Rearrange the model into a linear estimating equation in $(\theta_1,\theta_2)$.
4. Estimate $(\theta_1,\theta_2)$ by OLS.

#### Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd

np.random.seed(42)


#### Load Data


In [2]:
# Variables:
# i  = bus id
# t  = month
# a  = action (0 = maintain, 1 = replace)
# x  = mileage state in {1, ..., 7}
# rc = replacement cost
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"Months in panel: {df['t'].min()} to {df['t'].max()}")
print(f"Mileage states: {sorted(df['x'].unique().tolist())}")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)


Loaded 100,000 observations for 1,000 buses.
Months in panel: 1 to 100
Mileage states: [1, 2, 3, 4, 5, 6, 7]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


#### Global Constants


In [3]:
BETA = 0.95
N_X = 7
T_MAX = int(df["t"].max())

print(f"beta  = {BETA}")
print(f"N_X   = {N_X}")
print(f"T_MAX = {T_MAX}")


beta  = 0.95
N_X   = 7
T_MAX = 100


#### Usual discretization

For this simulated dataset, instead of discretizing and estimating an RC law of motion, we work directly with the observed time path (as usual)

In [4]:
# Check that RC_t is common across buses within each month
rc_counts_by_t = df.groupby("t")["rc"].nunique()
assert rc_counts_by_t.eq(1).all(), "RC must be common across buses within each month for this approach."

rc_by_t = df.groupby("t")["rc"].first().to_numpy()   # length 100

print("Unique RC values per t (should all equal 1):")
print(rc_counts_by_t.describe())
print(f"All equal to 1?  {rc_counts_by_t.eq(1).all()}")

print("\nFirst 15 values of the realized RC path:")
print(np.round(rc_by_t[:15], 3))


Unique RC values per t (should all equal 1):
count    100.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: rc, dtype: float64
All equal to 1?  True

First 15 values of the realized RC path:
[45.  49.  54.  47.  56.  52.5 55.  50.5 52.  42.  31.5 39.5 40.5 39.5
 42. ]


### Estimate $\hat P_t(a=1\mid x)$ as a Flexible Function of $x$

The homework note for part (d) says:

> For (d), you can estimate CCPs for each $t$ as a flexible function of $x$.

We follow that literally. Since $RC_t$ is common within a month, the relevant CCP for this approach is

$$
\hat P_t(a=1\mid x),
$$

estimated separately for each month $t$ as a flexible function of mileage $x$.

To keep logs well-defined and avoid zero-probability cells, we use a cubic logit smoother for each month:

$$
\Pr(a=1\mid x,t) = \Lambda(\alpha_{0t} + \alpha_{1t}x + \alpha_{2t}x^2 + \alpha_{3t}x^3),
$$

where

$$
\Lambda(z)=\frac{1}{1+e^{-z}}.
$$

We estimate each month's coefficient vector by Newton-Raphson / IRLS using explicit matrix algebra:

$$
g(\alpha_t)=Z_t^\top(a_t - p_t), \qquad
H(\alpha_t)=-Z_t^\top W_t Z_t,
$$

and update

$$
\alpha_t^{new} = \alpha_t - H(\alpha_t)^{-1} g(\alpha_t).
$$


It gives smoother CCPs than raw frequencies.

In [5]:
def fit_logit_cubic(x, a, maxiter=100, tol=1e-10):
    '''
    Fit logit Pr(a=1|x) = logistic(alpha0 + alpha1*x + alpha2*x^2 + alpha3*x^3)
    using Newton-Raphson / IRLS with explicit matrix algebra.

    Parameters
    ----------
    x : ndarray (n,)
    a : ndarray (n,)

    Returns
    -------
    alpha : ndarray (4,)
    '''
    Z = np.column_stack([
        np.ones_like(x, dtype=float),
        x.astype(float),
        x.astype(float) ** 2,
        x.astype(float) ** 3,
    ])  # shape (n, 4)

    alpha = np.zeros(Z.shape[1])

    for _ in range(maxiter):
        z = Z @ alpha
        z = np.clip(z, -30.0, 30.0)
        p = 1.0 / (1.0 + np.exp(-z))

        # IRLS objects
        w = np.clip(p * (1.0 - p), 1e-8, None)
        grad = Z.T @ (a - p)
        hess = -(Z.T * w) @ Z

        step = np.linalg.solve(hess, grad)
        alpha_new = alpha - step

        if np.max(np.abs(alpha_new - alpha)) < tol:
            alpha = alpha_new
            break
        alpha = alpha_new

    return alpha


t0_method = time.perf_counter()

# Time-by-mileage count matrices
N1_tx = np.zeros((T_MAX, N_X))
N0_tx = np.zeros((T_MAX, N_X))

for t, x, a in zip(df["t"].values.astype(int),
                   df["x"].values.astype(int),
                   df["a"].values.astype(int)):
    if a == 1:
        N1_tx[t - 1, x - 1] += 1
    else:
        N0_tx[t - 1, x - 1] += 1

N_tx = N0_tx + N1_tx
freq_p1_tx = np.where(N_tx > 0, N1_tx / N_tx, np.nan)

# Smoothed CCP matrix P1_hat_tx[t, x-1]
P1_hat_tx = np.zeros((T_MAX, N_X))
alpha_by_t = np.zeros((T_MAX, 4))
x_grid = np.arange(1, N_X + 1, dtype=float)
Z_grid = np.column_stack([
    np.ones(N_X),
    x_grid,
    x_grid ** 2,
    x_grid ** 3,
])  # shape (7, 4)

for t in range(1, T_MAX + 1):
    g = df.loc[df["t"] == t, ["x", "a"]]
    x_t = g["x"].to_numpy(dtype=float)
    a_t = g["a"].to_numpy(dtype=float)

    alpha_t = fit_logit_cubic(x_t, a_t)
    alpha_by_t[t - 1, :] = alpha_t

    p_t = 1.0 / (1.0 + np.exp(-np.clip(Z_grid @ alpha_t, -30.0, 30.0)))
    P1_hat_tx[t - 1, :] = np.clip(p_t, 1e-6, 1.0 - 1e-6)

P0_hat_tx = 1.0 - P1_hat_tx

print("Time-by-mileage count matrix N_tx shape:", N_tx.shape)
print("Smoothed CCP matrix P1_hat_tx shape:", P1_hat_tx.shape)

print("\nFirst 3 months: raw frequency CCP P(a=1|x,t)")
print(np.round(freq_p1_tx[:3], 3))

print("\nFirst 3 months: smoothed CCP P_hat(a=1|x,t)")
print(np.round(P1_hat_tx[:3], 3))


Time-by-mileage count matrix N_tx shape: (100, 7)
Smoothed CCP matrix P1_hat_tx shape: (100, 7)

First 3 months: raw frequency CCP P(a=1|x,t)
[[0.004 0.012 0.093 0.158 0.312 0.423 0.558]
 [0.    0.013 0.072 0.131 0.229 0.412 0.52 ]
 [0.    0.    0.03  0.065 0.144 0.365 0.484]]

First 3 months: smoothed CCP P_hat(a=1|x,t)
[[0.003 0.021 0.078 0.181 0.303 0.426 0.559]
 [0.002 0.014 0.058 0.142 0.252 0.377 0.536]
 [0.001 0.004 0.021 0.069 0.166 0.313 0.497]]


### Derive the Scott (2014) Linear Regression Equation

Let the current state be $(x_t, RC_t)$.

Under Type I EV errors, we can use the identity

$$
V(s) = -\ln P(a=1\mid s) + \delta_1(s) + \gamma,
$$

for any action label $1$, where

$$
\delta_1(s) = \bar u(s,1;\theta) + \beta EV(s,1;\theta).
$$

For the bus problem, action $1$ (replace) is the key "termination-action-style" choice because after replacing, the mileage next period resets to 1. Therefore,

$$
\delta_1(x,RC_t;\theta)
=
-\theta_2 RC_t + \beta V(1, RC_{t+1}),
$$

which does not depend on current mileage $x$ except through the common $RC_t$.

Now start from the current-period log-odds:

$$
\ln\!\left(\frac{P(a=0\mid x_t,RC_t)}{P(a=1\mid x_t,RC_t)}\right)
=
\delta_0(x_t,RC_t;\theta)-\delta_1(x_t,RC_t;\theta).
$$

Using the model payoffs,

$$
\delta_0(x_t,RC_t;\theta)
=
-\theta_1 x_t + \beta V(\min(x_t+1,7), RC_{t+1}),
$$

so

$$
\ln\!\left(\frac{P(a=0\mid x_t,RC_t)}{P(a=1\mid x_t,RC_t)}\right)
=
-\theta_1 x_t + \theta_2 RC_t
+ \beta\Bigl[
V(\min(x_t+1,7),RC_{t+1}) - V(1,RC_{t+1})
\Bigr].
$$

Now apply the Type-I-EV identity to the continuation states at $t+1$:

$$
V(x,RC_{t+1})
=
-\ln P(a=1\mid x,RC_{t+1}) + \delta_1(x,RC_{t+1}) + \gamma.
$$

Because $\delta_1(x,RC_{t+1})$ is the same for all $x$ once we condition on $RC_{t+1}$, it cancels in the difference, yielding

$$
V(\min(x_t+1,7),RC_{t+1}) - V(1,RC_{t+1})
=
-\ln P(a=1\mid \min(x_t+1,7),RC_{t+1})
+ \ln P(a=1\mid 1,RC_{t+1}).
$$

Substitute this into the log-odds equation:

$$
\ln\!\left(\frac{P(a=0\mid x_t,RC_t)}{P(a=1\mid x_t,RC_t)}\right)
+ \beta \ln\!\left(
\frac{P(a=1\mid \min(x_t+1,7),RC_{t+1})}
{P(a=1\mid 1,RC_{t+1})}
\right)
=
-\theta_1 x_t + \theta_2 RC_t.
$$

This is the Scott-style linear regression equation.

In our implementation, since $RC_t$ is common in each month, we write the empirical version as

$$
Y_{t,x}
=
\ln\!\left(\frac{\hat P_t(a=0\mid x)}{\hat P_t(a=1\mid x)}\right)
+ \beta \ln\!\left(
\frac{\hat P_{t+1}(a=1\mid \min(x+1,7))}
{\hat P_{t+1}(a=1\mid 1)}
\right)
=
-\theta_1 x + \theta_2 RC_t + u_{t,x}.
$$


In [6]:
log_odds_tx = np.log(P0_hat_tx) - np.log(P1_hat_tx)   # shape (100, 7)
log_p1_tx = np.log(P1_hat_tx)                         # shape (100, 7)

# Build state-level regression rows:
# one row for each (t, x) with t = 1,...,99
rows = []
for t_idx in range(T_MAX - 1):   # 0..98  <=>  t=1..99
    for x_idx in range(N_X):     # 0..6   <=>  x=1..7
        n_tx = N_tx[t_idx, x_idx]
        if n_tx <= 0:
            continue

        x = x_idx + 1
        x_next = min(x + 1, N_X)
        rc_t = rc_by_t[t_idx]

        y_tx = (
            log_odds_tx[t_idx, x_idx]
            + BETA * (log_p1_tx[t_idx + 1, x_next - 1] - log_p1_tx[t_idx + 1, 0])
        )

        rows.append((t_idx + 1, x, rc_t, n_tx, y_tx))

reg_df = pd.DataFrame(rows, columns=["t", "x", "rc", "weight", "y"])

print("Regression dataset head:")
print(reg_df.head(10))
print(f"\nNumber of state-level rows: {len(reg_df):,}")
print(f"Total weight (agent-period observations used): {reg_df['weight'].sum():,.0f}")


Regression dataset head:
   t  x    rc  weight         y
0  1  1  45.0   235.0  7.817618
1  1  2  45.0   168.0  7.203823
2  1  3  45.0   194.0  6.673091
3  1  4  45.0   114.0  6.254182
4  1  5  45.0   141.0  5.964445
5  1  6  45.0    71.0  5.765869
6  1  7  45.0    77.0  5.230818
7  2  1  49.0   156.0  8.350329
8  2  2  49.0   234.0  7.755339
9  2  3  49.0   166.0  7.425475

Number of state-level rows: 693
Total weight (agent-period observations used): 99,000


### Estimate $(\theta_1,\theta_2)$ by Weighted OLS

The regression equation is

$$
Y_{t,x} = -\theta_1 x + \theta_2 RC_t + u_{t,x}.
$$

Write this in matrix form:

$$
\mathbf{y} = X\beta + u,
\qquad
\beta =
\begin{bmatrix}
b_x \\
b_{RC}
\end{bmatrix}
=
\begin{bmatrix}
-\theta_1 \\
\theta_2
\end{bmatrix}.
$$

We use state-level rows indexed by $(t,x)$, but weight each row by the number of observations in that cell:

$$
W = \mathrm{diag}(n_{t,x}).
$$

Then weighted OLS is

$$
\hat\beta = (X^\top W X)^{-1} X^\top W \mathbf{y}.
$$

This is equivalent to observation-level OLS with one-agent-period-one-vote, because repeating the same state-level row $n_{t,x}$ times gives the same normal equations.


In [7]:
# Weighted OLS with no intercept:
# y = b_x * x + b_rc * rc + error
# where theta1 = -b_x and theta2 = b_rc

t0_theta = time.perf_counter()
y = reg_df["y"].to_numpy()
X = np.column_stack([
    reg_df["x"].to_numpy(),
    reg_df["rc"].to_numpy(),
])  # shape (n_rows, 2)
w = reg_df["weight"].to_numpy()

XtWX = X.T @ (w[:, None] * X)
XtWy = X.T @ (w * y)
beta_hat = np.linalg.solve(XtWX, XtWy)

b_x_hat, b_rc_hat = beta_hat
theta1_hat = -b_x_hat
theta2_hat = b_rc_hat

# Simple frequency-weighted OLS variance formula
resid = y - X @ beta_hat
n_obs_equiv = int(w.sum())
k = X.shape[1]
sigma2_hat = (w * resid ** 2).sum() / (n_obs_equiv - k)
vcov_hat = sigma2_hat * np.linalg.inv(XtWX)
se_beta = np.sqrt(np.diag(vcov_hat))
se_theta1 = se_beta[0]
se_theta2 = se_beta[1]

# Weighted R^2
y_bar_w = np.average(y, weights=w)
ssr = np.sum(w * resid ** 2)
sst = np.sum(w * (y - y_bar_w) ** 2)
r2_w = 1.0 - ssr / sst
theta_runtime_sec = time.perf_counter() - t0_theta
theta_runtime_ms = 1000.0 * theta_runtime_sec
running_time_sec = time.perf_counter() - t0_method
running_time_ms = 1000.0 * running_time_sec

print("Weighted OLS coefficients [b_x, b_rc]:")
print(np.round(beta_hat, 6))
print("\nImplied theta estimates:")
print(f"theta1_hat = {-b_x_hat:.6f}")
print(f"theta2_hat = {b_rc_hat:.6f}")
print("\nApproximate standard errors:")
print(f"se(theta1_hat) = {se_theta1:.6f}")
print(f"se(theta2_hat) = {se_theta2:.6f}")
print(f"Weighted R^2    = {r2_w:.6f}")
print(f"OLS-only time   = {theta_runtime_sec:.4f} seconds ({theta_runtime_ms:.2f} ms)")
print(f"Comparable running time = {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")


Weighted OLS coefficients [b_x, b_rc]:
[-0.473937  0.172606]

Implied theta estimates:
theta1_hat = 0.473937
theta2_hat = 0.172606

Approximate standard errors:
se(theta1_hat) = 0.002443
se(theta2_hat) = 0.000210
Weighted R^2    = 0.459507
OLS-only time   = 0.0008 seconds (0.78 ms)
Comparable running time = 0.1171 seconds (117.12 ms)


### Weighting Interpretation

Here we use state-level weighted OLS, with one row per $(t,x)$ cell and weight equal to the number of bus-month observations in that cell.

So the estimator is numerically equivalent to observation-level OLS with one-agent-period-one-vote:

- cells with more observations get more weight,
- rare $(t,x)$ cells get less weight,
- this matches the implicit weighting in Parts (a), (b), and (c), where each observation contributes to the criterion.


In [8]:
# Verify equivalence with observation-level OLS:
# merge the state-level y_{t,x} back to each observation and run plain OLS
obs_df = df.merge(reg_df[["t", "x", "y"]], on=["t", "x"], how="left")
obs_df = obs_df[obs_df["t"] <= T_MAX - 1].copy()   # only t=1,...,99 have continuation terms

X_obs = np.column_stack([
    obs_df["x"].to_numpy(),
    obs_df["rc"].to_numpy(),
])
y_obs = obs_df["y"].to_numpy()

beta_hat_obs = np.linalg.solve(X_obs.T @ X_obs, X_obs.T @ y_obs)

print("Observation-level OLS coefficients [b_x, b_rc]:")
print(np.round(beta_hat_obs, 6))
print("\nDifference from weighted state-level OLS:")
print(np.round(beta_hat_obs - beta_hat, 12))


Observation-level OLS coefficients [b_x, b_rc]:
[-0.473937  0.172606]

Difference from weighted state-level OLS:
[ 0. -0.]


### Results


In [9]:
print("=" * 62)
print("        PART (d) -- SCOTT (2014) LINEAR REGRESSION")
print("=" * 62)
print(f"theta1_hat    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"theta2_hat    (replacement cost coeff.) = {theta2_hat:.4f}")
print("-" * 62)
print(f"Weighted R^2                              = {r2_w:.4f}")
print(f"State-level rows used                     = {len(reg_df):,}")
print(f"Observation-equivalent sample size        = {n_obs_equiv:,}")
print(f"OLS-only time                             = {theta_runtime_sec:.4f} sec ({theta_runtime_ms:.2f} ms)")
print(f"Comparable running time                   = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("-" * 62)
print("Interpretation:")
print("  coefficient on x   = -theta1")
print("  coefficient on RC  =  theta2")
print("=" * 62)


        PART (d) -- SCOTT (2014) LINEAR REGRESSION
theta1_hat    (mileage cost coeff.)     = 0.4739
theta2_hat    (replacement cost coeff.) = 0.1726
--------------------------------------------------------------
Weighted R^2                              = 0.4595
State-level rows used                     = 693
Observation-equivalent sample size        = 99,000
OLS-only time                             = 0.0008 sec (0.78 ms)
Comparable running time                   = 0.1171 sec (117.12 ms)
--------------------------------------------------------------
Interpretation:
  coefficient on x   = -theta1
  coefficient on RC  =  theta2


### Summary

- We do not solve a Bellman equation.
- We do not invert a Hotz-Miller linear system.
- We do not simulate forward to approximate the value function.
- We do not estimate the RC transition process explicitly.

Instead, we use the special structure of the replacement action:

- replacing resets mileage to 1,
- so the continuation value difference can be rewritten using observable CCP differences,
- which turns the model into a linear regression in $(\theta_1,\theta_2)$.

**Note:** This estimator depends heavily on the quality of the CCP estimates:
- if some $(t,x)$ cells are sparse,
- or if the estimated CCPs are noisy,
- then the log terms can become unstable.

That is why we used a smooth cubic-logit CCP estimator rather than raw unsmoothed frequencies.
